# Instacart — KPI Analysis and Visualization

This notebook uses the analysis tables created previously to:

- analyze order activity;
- study customer behavior;
- identify the best-performing products and categories;
- analyze temporal purchasing patterns;
- create visualizations that can be used in a dashboard.

In [0]:
# ---------------------------------------------------------
# Step 1 — Load Analysis Tables
# ---------------------------------------------------------

from pyspark.sql import functions as F

# KPI by commande
order_kpis_df = spark.table(
    "workspace.analysis_data.order_kpis"
)

# KPI by client
customer_kpis_df = spark.table(
    "workspace.analysis_data.customer_kpis"
)

# KPI by produit
product_kpis_df = spark.table(
    "workspace.analysis_data.product_kpis"
)

# KPI by allée
aisle_kpis_df = spark.table(
    "workspace.analysis_data.aisle_kpis"
)

# KPI by département
department_kpis_df = spark.table(
    "workspace.analysis_data.department_kpis"
)

# Temporary KPI 
daily_kpis_df = spark.table(
    "workspace.analysis_data.daily_kpis"
)

hourly_kpis_df = spark.table(
    "workspace.analysis_data.hourly_kpis"
)

time_period_kpis_df = spark.table(
    "workspace.analysis_data.time_period_kpis"
)

print("Les huit tables d’analyse ont été chargées avec succès.")

Les huit tables d’analyse ont été chargées avec succès.


In [0]:
# ---------------------------------------------------------
# Step 2 — Verify charged tables
# ---------------------------------------------------------

analysis_table_counts = [
    ("order_kpis", order_kpis_df.count()),
    ("customer_kpis", customer_kpis_df.count()),
    ("product_kpis", product_kpis_df.count()),
    ("aisle_kpis", aisle_kpis_df.count()),
    ("department_kpis", department_kpis_df.count()),
    ("daily_kpis", daily_kpis_df.count()),
    ("hourly_kpis", hourly_kpis_df.count()),
    ("time_period_kpis", time_period_kpis_df.count())
]

analysis_table_counts_df = spark.createDataFrame(
    analysis_table_counts,
    ["table_name", "row_count"]
)

display(
    analysis_table_counts_df.orderBy("table_name")
)

table_name,row_count
aisle_kpis,134
customer_kpis,206209
daily_kpis,7
department_kpis,21
hourly_kpis,24
order_kpis,3346083
product_kpis,49688
time_period_kpis,4


## 2. Product Analysis

### 2.1 Most Purchased Products

This analysis identifies the products with the highest purchase volume in order to highlight the most popular products among customers.

In [0]:
# ---------------------------------------------------------
# 2.1 — Top 20 Most Purchased Products
# ---------------------------------------------------------

top_products_df = (
    product_kpis_df
    .select(
        "product_id",
        "product_name",
        "aisle",
        "department",
        "total_purchase_count",
        "unique_customer_count",
        "product_reorder_rate"
    )
    .orderBy(
        F.desc("total_purchase_count")
    )
    .limit(20)
)

display(top_products_df)

product_id,product_name,aisle,department,total_purchase_count,unique_customer_count,product_reorder_rate
24852,Banana,fresh fruits,produce,491291,76125,0.8451
13176,Bag of Organic Bananas,fresh fruits,produce,394930,65655,0.8338
21137,Organic Strawberries,fresh fruits,produce,275577,61129,0.7782
21903,Organic Baby Spinach,packaged vegetables fruits,produce,251705,56766,0.7745
47209,Organic Hass Avocado,fresh fruits,produce,220877,44704,0.7976
47766,Organic Avocado,fresh fruits,produce,184224,43954,0.7614
47626,Large Lemon,fresh fruits,produce,160792,48614,0.6977
16797,Strawberries,fresh fruits,produce,149445,44857,0.6998
26209,Limes,fresh fruits,produce,146660,46658,0.6819
27845,Organic Whole Milk,milk,dairy eggs,142813,24129,0.831


Databricks visualization. Run in Databricks to view.

### 2.2 Product Popularity vs. Reorder Rate

In [0]:
# ---------------------------------------------------------
# 2.2 — Product Popularity vs. Reorder Rate
# ---------------------------------------------------------

product_loyalty_df = (
    product_kpis_df

    # Exclude products with too few observations
    .filter(
        F.col("total_purchase_count") >= 1000
    )

    # Keep the highest-volume products
    .orderBy(
        F.desc("total_purchase_count")
    )
    .limit(200)

    .select(
        "product_id",
        "product_name",
        "department",
        "total_purchase_count",
        "product_reorder_rate"
    )

    .withColumn(
        "reorder_rate_pct",
        F.round(F.col("product_reorder_rate") * 100, 2)
    )
)

display(product_loyalty_df)

product_id,product_name,department,total_purchase_count,product_reorder_rate,reorder_rate_pct
24852,Banana,produce,491291,0.8451,84.51
13176,Bag of Organic Bananas,produce,394930,0.8338,83.38
21137,Organic Strawberries,produce,275577,0.7782,77.82
21903,Organic Baby Spinach,produce,251705,0.7745,77.45
47209,Organic Hass Avocado,produce,220877,0.7976,79.76
47766,Organic Avocado,produce,184224,0.7614,76.14
47626,Large Lemon,produce,160792,0.6977,69.77
16797,Strawberries,produce,149445,0.6998,69.98
26209,Limes,produce,146660,0.6819,68.19
27845,Organic Whole Milk,dairy eggs,142813,0.831,83.1


Databricks visualization. Run in Databricks to view.

## 3. Category Analysis

### 3.1 Department Analysis

This analysis compares departments based on their total purchase volume in order to identify the main product categories driving Instacart activity.

In [0]:
# ---------------------------------------------------------
# 3.1 — Department Ranking by Purchase Volume
# ---------------------------------------------------------

top_departments_df = (
    department_kpis_df
    .select(
        "department_id",
        "department",
        "total_purchase_count"
    )
    .orderBy(
        F.desc("total_purchase_count")
    )
)

display(top_departments_df)

department_id,department,total_purchase_count
4,produce,9888378
16,dairy eggs,5631067
19,snacks,3006412
7,beverages,2804175
1,frozen,2336858
13,pantry,1956819
3,bakery,1225181
15,canned goods,1114857
20,deli,1095540
9,dry goods pasta,905340


Databricks visualization. Run in Databricks to view.

### 3.2 Aisle Analysis

This analysis identifies the top 20 aisles by purchase volume in order to highlight the most important product subcategories within Instacart activity.

In [0]:
# ---------------------------------------------------------
# 3.2 — Top 20 Aisles by Purchase Volume
# ---------------------------------------------------------

top_aisles_df = (
    aisle_kpis_df
    .select(
        "aisle_id",
        "aisle",
        "total_purchase_count"
    )
    .orderBy(
        F.desc("total_purchase_count")
    )
    .limit(20)
)

display(top_aisles_df)

aisle_id,aisle,total_purchase_count
24,fresh fruits,3792661
83,fresh vegetables,3568630
123,packaged vegetables fruits,1843806
120,yogurt,1507583
21,packaged cheese,1021462
84,milk,923659
115,water seltzer sparkling water,878150
107,chips pretzels,753739
91,soy lactosefree,664493
112,bread,608469


Databricks visualization. Run in Databricks to view.

## 4. Customer Analysis

### 4.1 Order Frequency

This analysis examines the distribution of the number of orders per customer in order to understand purchasing frequency and identify the most common ordering behaviors.

In [0]:
# ---------------------------------------------------------
# 4.1 — Distribution of Orders per Customer
# ---------------------------------------------------------

customer_order_frequency_df = (
    customer_kpis_df
    .groupBy("total_order_count")
    .agg(
        F.count("user_id").alias("customer_count")
    )
    .orderBy("total_order_count")
)

display(customer_order_frequency_df)

total_order_count,customer_count
4,23986
5,19590
6,16165
7,13850
8,11700
9,10190
10,9032
11,7815
12,6952
13,6236


Databricks visualization. Run in Databricks to view.

### 4.2 Customer Segmentation

This analysis shows the distribution of customers across purchase-frequency segments in order to identify the most represented customer profiles.

In [0]:
# ---------------------------------------------------------
# 4.2 — Customer Distribution by Segment
# ---------------------------------------------------------

customer_segments_df = (
    customer_kpis_df
    .groupBy("customer_frequency_segment")
    .agg(
        F.count("user_id").alias("customer_count")
    )
    .orderBy(
        F.desc("customer_count")
    )
)

display(customer_segments_df)

customer_frequency_segment,customer_count
regulier,92744
fidele,69889
occasionnel,43576


Databricks visualization. Run in Databricks to view.

## 5. Temporal Analysis

### 5.1 Orders by Day of Week

This analysis examines the distribution of orders across the days of the week in order to identify the days with the highest activity.

In [0]:
# ---------------------------------------------------------
# 5.1 — Number of Orders by Day of Week
# ---------------------------------------------------------

daily_orders_df = (
    daily_kpis_df
    .select(
        "order_dow",
        "total_order_count"
    )
    .orderBy("order_dow")
)

display(daily_orders_df)

order_dow,total_order_count
0,585237
1,576377
2,458074
3,428087
4,417171
5,443388
6,437749


Databricks visualization. Run in Databricks to view.

### 5.2 Orders by Hour of Day

This analysis examines how the number of orders changes throughout the day in order to identify peak and low-activity hours.

In [0]:
# ---------------------------------------------------------
# 5.2 — Number of Orders by Hour of Day
# ---------------------------------------------------------

hourly_orders_df = (
    hourly_kpis_df
    .select(
        "order_hour_of_day",
        "total_order_count"
    )
    .orderBy("order_hour_of_day")
)

display(hourly_orders_df)

order_hour_of_day,total_order_count
0,22224
1,12103
2,7375
3,5343
4,5393
5,9374
6,29913
7,90032
8,174664
9,252529


Databricks visualization. Run in Databricks to view.

### 5.3 Orders by Time Period

This analysis compares order volume across different periods of the day in order to identify when activity is highest.

In [0]:
# ---------------------------------------------------------
# 5.3 — Number of Orders by Time Period
# ---------------------------------------------------------

time_period_orders_df = (
    time_period_kpis_df
    .select(
        "order_time_period",
        "total_order_count"
    )
    .orderBy(
        F.desc("total_order_count")
    )
)

display(time_period_orders_df)

order_time_period,total_order_count
afternoon,1359023
morning,1117598
evening,717903
night,151559


Databricks visualization. Run in Databricks to view.

## 6. Key Insights

The KPI analysis highlights several important patterns in Instacart customers' purchasing behavior:

- **Fresh products dominate purchases.** Banana is the most purchased product, with 491,291 purchases, followed by Bag of Organic Bananas. The `fresh fruits` and `fresh vegetables` aisles are also among the highest-volume aisles.

- **The `produce` department accounts for the highest purchase volume**, significantly ahead of the other departments, followed notably by `dairy eggs`, `snacks`, and `beverages`.

- **Several products show strong customer loyalty.** Among products with at least 1,000 purchases, several have a reorder rate above 80%, including a number of dairy products.

- **Most customers place relatively few orders**, while a smaller group orders much more frequently. The `regular` customer segment is the largest, followed by `loyal` and then `occasional` customers.

- **Order activity varies significantly throughout the day.** Orders increase sharply in the morning, remain high between approximately 10 AM and 3 PM, and then gradually decline. Overall, the afternoon is the busiest period, while nighttime activity is considerably lower.

- **Days 0 and 1 record the highest order volumes in the dataset's day-of-week coding**, highlighting clear differences in activity depending on the day of the week.